# Group 2: Luca Milani, Marta Laskowska, Monika Kaczorowska

### Libraries

In [1]:
import pandas as pd
import requests
import bs4 as bs
import yfinance as yf
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
import re
import html

pd.set_option("display.max_colwidth", None)


### Dataset

---
### Task #4 : Measuring media attention (1)

• Use the list of tickers gathered during last PC Lab (see the web-scrapping part) to compute the number of tweets about each stock

– e.g., AAPL: 36 tweets, 12 negative, 24 positive

With the new dataset provided for the assignment, answering the following question required no web scraping; therefore, we proceed without the step of scraping the list of tickers.

In [2]:
# Loading the aggregated sentiment scores
df_aggregated_sentiments = pd.read_csv("final_aggregated_sentiments.csv")
df_aggregated_sentiments.sort_values(by="date", ascending=False).reset_index(drop=True)

,ticker,date,n_tweets,sent_index
0,AMZN,2020-07-22,706,0.2
1,BAC,2020-07-22,51,0.1
2,NFLX,2020-07-22,37,-0.1
3,WMT,2020-07-21,27,-0.2
4,NFLX,2020-07-21,354,-0.2
...,...,...,...,...
52457,AAPL,2009-07-29,1,1.0
52458,BAC,2009-07-13,1,1.0
52459,AAPL,2009-07-13,3,0.0
52460,NFLX,2009-07-13,1,1.0


In [3]:
df_aggregated_sentiments.describe()

,n_tweets,sent_index
count,52462.000000,52462.000000
mean,88.599005,0.138553
std,289.331361,0.289658
min,1.000000,-1.000000
25%,5.000000,0.000000
50%,17.000000,0.100000
75%,60.000000,0.300000
max,14971.000000,1.000000


#### Rank the stocks by their amount of total media attention, or, alternatively : positive and negative media attention, level of disagreement (dispersion), etc.

In [5]:
attention_rank = (
    df_aggregated_sentiments.groupby("ticker")["n_tweets"]
    .sum()
    .sort_values(ascending=False)
)
attention_rank

ticker
TSLA     1514647
AAPL     1101533
NFLX      585224
AMZN      397958
DIS       212834
BAC       147407
GOOG      124147
GOOGL      99132
FB         96800
T          62712
HD         46981
VZ         45889
V          44606
JNJ        34615
ADBE       33014
BRK.B      30124
PG         29625
UNH        14428
INTC       13920
WMT        12485
Name: n_tweets, dtype: int64

The Top 5 stocks with the greatest number of tweets over the researched period (after having applied constraints to the length of the tweet) are: 
- Tesla, 
- Apple,
- Netflix,
- Amazon and
- Walt Disney Co

In [6]:
# total tweets per ticker
total_per_ticker = df_aggregated_sentiments.groupby("ticker")["n_tweets"].sum()

# positive and negative tweets per ticker
pos_per_ticker = df_aggregated_sentiments.loc[df_aggregated_sentiments["sent_index"] > 0].groupby("ticker")["n_tweets"].sum()
neg_per_ticker = df_aggregated_sentiments.loc[df_aggregated_sentiments["sent_index"] < 0].groupby("ticker")["n_tweets"].sum()

# normalize: share of positive/negative
pos_share = (pos_per_ticker / total_per_ticker).fillna(0).sort_values(ascending=False)
neg_share = (neg_per_ticker / total_per_ticker).fillna(0).sort_values(ascending=False)

print("Top 10 by *share* of positive attention:\n", pos_share.head(10))
print("\nTop 10 by *share* of negative attention:\n", neg_share.head(10))


Top 10 by *share* of positive attention:
 ticker
V        0.715061
GOOG     0.709739
ADBE     0.704610
GOOGL    0.697040
VZ       0.648325
INTC     0.635991
JNJ      0.625856
HD       0.614142
AMZN     0.610466
WMT      0.608971
Name: n_tweets, dtype: float64

Top 10 by *share* of negative attention:
 ticker
TSLA     0.443773
NFLX     0.350204
FB       0.299370
BAC      0.279695
DIS      0.267373
AAPL     0.265570
AMZN     0.242485
BRK.B    0.230248
T        0.206595
HD       0.197931
Name: n_tweets, dtype: float64


Companies with the **highest positive attention** are those often associated with **strong brands, innovation** or c**onsistent financial performance**. On the other hand, companies that lead the **negative sentiment** rankings often spark polarizing opinions due to **leadership controversies, aggressive strategies** or **volatile performance**.

In [ ]:
disagreement_rank = (
    df_aggregated_sentiments.groupby("ticker")["sent_index"]
    .std()  # high std = more dispersion in daily sentiment
    .sort_values(ascending=False)
)

disagreement_rank

ticker
UNH      0.355130
WMT      0.349057
INTC     0.339239
ADBE     0.337469
HD       0.315216
VZ       0.310314
BRK.B    0.309564
FB       0.307352
DIS      0.299169
JNJ      0.290081
PG       0.289719
TSLA     0.289432
T        0.286627
V        0.268366
NFLX     0.251349
BAC      0.248856
AAPL     0.242661
AMZN     0.242657
GOOGL    0.233982
GOOG     0.230000
Name: sent_index, dtype: float64

With the contraints applied to the tweets (the length of the tweet and the random sampling of 10 tweets per day per stock), the **stocks with the greatest level of disagreement**, indicated by the greatest dispersion of the sentiment score, are:
  - UnitedHealth Group Inc
  - Walmart Inc
  - Intel Corp
  - Adobe Inc
  - Home Depot Inc

The high dispersion in sentiment scores of those comapnies can be attributed to the **diversity of their stakeholders** and the **nature of the industries**. Firms like UnitedHealth and Walmart operate in sectors that directly affect **broad groups of people**, while technology companies such as Intel and Adobe are subject to polarized reactions due to **competitive dynamics** and the **pace of innovation** which can spark both enthusiasm and skepticism. In addition, strongly **consumer-facing businesses** like Walmart and Home Depot, with customer experiences, pricing and service quality, often generate both praise and criticism. These factors explain why such companies exhibit greater disagreement in sentiment.

**Media Attention Measure**

$$
Media\_Attention\_Score = \log(1 + n\_tweets) \times sent\_index 
$$

**Logarithm transformation** controls for diminishing marginal effect of tweet volume, while smoothing the dynamics. 

#### • Create 10 portfolios based on your preferred measure of media attention

In [7]:
df_aggregated_sentiments.head()

,ticker,date,n_tweets,sent_index
0,AAPL,2009-07-10,1,0.0
1,AAPL,2009-07-13,3,0.0
2,AAPL,2009-07-29,1,1.0
3,AAPL,2009-07-30,1,0.0
4,AAPL,2009-07-31,2,-0.5


#### • Do you see a correlation between media attention g and stock returns?

#### • If yes, could Twitter attention is likely to be a good factor?

#### • Optional : same, but use FF-5 factors to purge returns from what a traditional model predicts, and plot media attention vs. erros

#### Download the Fama–French 5 factors (Mkt-RF, SMB, HML, RMW, CMA, and RF) from Ken French’s Data Library

#### • Regress the stock’s excess returns on these five factors

####  Save the residuals from this regression. These represent the component of returns unexplained by traditional risk factors (abnormal returns).

#### • Redo same plot (see previous slide), but residuals vs. media attention.